In [44]:
import numpy as np
import pandas as pd
from pathlib import Path

In [45]:
csv_path = Path(r"C:\Users\aynur\Downloads\archive (8)\SeoulBikeData.csv")
df = pd.read_csv(csv_path, encoding="latin1")
df.head()

,Date,Rented Bike Count,Hour,Temperature(°C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(°C),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons,Holiday,Functioning Day
0,01/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,01/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,01/12/2017,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,01/12/2017,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,01/12/2017,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes


In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Date                       8760 non-null   object 
 1   Rented Bike Count          8760 non-null   int64  
 2   Hour                       8760 non-null   int64  
 3   Temperature(°C)            8760 non-null   float64
 4   Humidity(%)                8760 non-null   int64  
 5   Wind speed (m/s)           8760 non-null   float64
 6   Visibility (10m)           8760 non-null   int64  
 7   Dew point temperature(°C)  8760 non-null   float64
 8   Solar Radiation (MJ/m2)    8760 non-null   float64
 9   Rainfall(mm)               8760 non-null   float64
 10  Snowfall (cm)              8760 non-null   float64
 11  Seasons                    8760 non-null   object 
 12  Holiday                    8760 non-null   object 
 13  Functioning Day            8760 non-null   objec

In [47]:
df.isnull().sum()

Date                         0
Rented Bike Count            0
Hour                         0
Temperature(°C)              0
Humidity(%)                  0
Wind speed (m/s)             0
Visibility (10m)             0
Dew point temperature(°C)    0
Solar Radiation (MJ/m2)      0
Rainfall(mm)                 0
Snowfall (cm)                0
Seasons                      0
Holiday                      0
Functioning Day              0
dtype: int64

In [48]:
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y")
df["day_of_week"] = df["Date"].dt.dayofweek

In [49]:
df['month'] = df['Date'].dt.month

In [ ]:

df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x in [5, 6] else 0)

In [51]:
df

,Date,Rented Bike Count,Hour,Temperature(°C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(°C),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons,Holiday,Functioning Day,day_of_week,month,is_weekend
0,2017-12-01,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes,4,12,0
1,2017-12-01,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes,4,12,0
2,2017-12-01,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes,4,12,0
3,2017-12-01,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes,4,12,0
4,2017-12-01,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes,4,12,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2018-11-30,1003,19,4.2,34,2.6,1894,-10.3,0.0,0.0,0.0,Autumn,No Holiday,Yes,4,11,0
8756,2018-11-30,764,20,3.4,37,2.3,2000,-9.9,0.0,0.0,0.0,Autumn,No Holiday,Yes,4,11,0
8757,2018-11-30,694,21,2.6,39,0.3,1968,-9.9,0.0,0.0,0.0,Autumn,No Holiday,Yes,4,11,0
8758,2018-11-30,712,22,2.1,41,1.0,1859,-9.8,0.0,0.0,0.0,Autumn,No Holiday,Yes,4,11,0


In [ ]:
import numpy as np
import pandas as pd



df['hour_sin'] = np.sin(2 * np.pi * df['Hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['Hour'] / 24)



In [53]:
df.drop('Hour', axis = 1, inplace = True)

In [54]:
X = df.drop('Rented Bike Count', axis = 'columns')
y = df['Rented Bike Count']

In [55]:
num_cols =  X.select_dtypes(include = 'number').columns.to_list()
cat_cols = X.select_dtypes(include = 'object').columns.to_list()


In [56]:
num_cols

['Temperature(°C)',
 'Humidity(%)',
 'Wind speed (m/s)',
 'Visibility (10m)',
 'Dew point temperature(°C)',
 'Solar Radiation (MJ/m2)',
 'Rainfall(mm)',
 'Snowfall (cm)',
 'day_of_week',
 'month',
 'is_weekend',
 'hour_sin',
 'hour_cos']

In [57]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

num_pipeline = Pipeline([
    ('scaling', StandardScaler())
])

cat_pipeline = Pipeline([
    ('encoding', OneHotEncoder())
])

preprocessing = ColumnTransformer([
    ('CAT', cat_pipeline, cat_cols), 
    ('NUM', num_pipeline, num_cols)
])

X_arr = preprocessing.fit_transform(X)
X_prepared = pd.DataFrame(X_arr, columns = preprocessing.get_feature_names_out())

In [58]:
X_prepared

,CAT__Seasons_Autumn,CAT__Seasons_Spring,CAT__Seasons_Summer,CAT__Seasons_Winter,CAT__Holiday_Holiday,CAT__Holiday_No Holiday,CAT__Functioning Day_No,CAT__Functioning Day_Yes,NUM__Temperature(°C),NUM__Humidity(%),...,NUM__Visibility (10m),NUM__Dew point temperature(°C),NUM__Solar Radiation (MJ/m2),NUM__Rainfall(mm),NUM__Snowfall (cm),NUM__day_of_week,NUM__month,NUM__is_weekend,NUM__hour_sin,NUM__hour_cos
0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,-1.513957,-1.042483,...,0.925871,-1.659605,-0.655132,-0.1318,-0.171891,0.499144,1.587648,-0.631243,2.595313e-17,1.414214
1,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,-1.539074,-0.993370,...,0.925871,-1.659605,-0.655132,-0.1318,-0.171891,0.499144,1.587648,-0.631243,3.660254e-01,1.366025
2,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,-1.580936,-0.944257,...,0.925871,-1.667262,-0.655132,-0.1318,-0.171891,0.499144,1.587648,-0.631243,7.071068e-01,1.224745
3,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,-1.597680,-0.895144,...,0.925871,-1.659605,-0.655132,-0.1318,-0.171891,0.499144,1.587648,-0.631243,1.000000e+00,1.000000
4,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,-1.580936,-1.091596,...,0.925871,-1.736177,-0.655132,-0.1318,-0.171891,0.499144,1.587648,-0.631243,1.224745e+00,0.707107
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,-0.726961,-1.189822,...,0.751605,-1.100630,-0.655132,-0.1318,-0.171891,0.499144,1.297612,-0.631243,-1.366025e+00,0.366025
8756,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,-0.793939,-1.042483,...,0.925871,-1.070001,-0.655132,-0.1318,-0.171891,0.499144,1.297612,-0.631243,-1.224745e+00,0.707107
8757,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,-0.860918,-0.944257,...,0.873263,-1.070001,-0.655132,-0.1318,-0.171891,0.499144,1.297612,-0.631243,-1.000000e+00,1.000000
8758,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,-0.902779,-0.846031,...,0.694064,-1.062344,-0.655132,-0.1318,-0.171891,0.499144,1.297612,-0.631243,-7.071068e-01,1.224745


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_prepared, y, test_size = 0.2)

In [61]:
X_train['is_raining'] = (X_train['Rainfall(mm)'] > 0).astype(int)
X_train['is_snowing'] = (X_train['Snowfall (cm)'] > 0).astype(int)
X_train['temp_hum_idx'] = X_train['Temperature(°C)'] * X_train['Humidity(%)']


In [ ]:
from sklearn.ensemble import RandomForestRegressor
full_pipeline = Pipeline([
    ('cleaning', preprocessing),
    ('model', RandomForestRegressor()) 
])

In [ ]:
import numpy as np


full_pipeline.fit(X_train, np.log1p(y_train))

y_train_predict = np.expm1(full_pipeline.predict(X_train))
y_test_predict = np.expm1(full_pipeline.predict(X_test))

In [ ]:
# from sklearn.model_selection import RandomizedSearchCV
# from sklearn.metrics import root_mean_squared_log_error


# param_dist = {
#     'model__n_estimators': [500, 800, 1000],
#     'model__max_depth': [20, 30, 50, None],
#     'model__min_samples_leaf': [1, 2],
#     'model__max_features': [0.7, 'sqrt', None],
#     'model__bootstrap': [True]
# }

# random_search = RandomizedSearchCV(
#     full_pipeline, 
#     param_distributions=param_dist, 
#     n_iter=15, 
#     cv=5, 
#     scoring='neg_mean_squared_log_error',
#     verbose=1, 
#     n_jobs=-1,
#     random_state=42
# )

# random_search.fit(X_train, y_train)
# best_rf = random_search.best_estimator_


# train_rmsle = root_mean_squared_log_error(y_train, best_rf.predict(X_train))
# test_rmsle = root_mean_squared_log_error(y_test, best_rf.predict(X_test))


# print(f"Ən yaxşı parametrlər: {random_search.best_params_}")
# print(f"TRAIN RMSLE: {train_rmsle:.4f}")
# print(f"TEST RMSLE: {test_rmsle:.4f}")

In [ ]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_log_error

xgb_model = XGBRegressor(
    n_estimators=5000, 
    learning_rate=0.01,
    max_depth=5,              
    min_child_weight=20,      
    gamma=1,                  
    subsample=0.8,            
    colsample_bytree=0.6,     
    reg_alpha=5,              
    reg_lambda=10,            
    early_stopping_rounds=50, 
    random_state=42
)


regressor = TransformedTargetRegressor(
    regressor=xgb_model,
    func=np.log1p,
    inverse_func=np.expm1
)


xgb_pipeline_final = Pipeline([
    ('cleaning', preprocessing), 
    ('model', regressor)
])


X_test_transformed = preprocessing.transform(X_test)
y_test_log = np.log1p(y_test)



xgb_pipeline_final.fit(
    X_train, y_train,
    model__eval_set=[(X_test_transformed, y_test_log)],
    model__verbose=200 
)


train_pred = xgb_pipeline_final.predict(X_train)
test_pred = xgb_pipeline_final.predict(X_test)

train_rmsle = root_mean_squared_log_error(y_train, train_pred)
test_rmsle = root_mean_squared_log_error(y_test, test_pred)

print("\n" + "="*40)
print(f"YEKUN TRAIN RMSLE: {train_rmsle:.4f}")
print(f"YEKUN TEST RMSLE:  {test_rmsle:.4f}")
print("="*40)


importances = xgb_pipeline_final.named_steps['model'].regressor_.feature_importances_
feature_names = xgb_pipeline_final.named_steps['cleaning'].get_feature_names_out()
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)

print("\nƏn vacib 10 faktor:")
print(feat_imp.head(10))

Model öyrədilir, zəhmət olmasa gözləyin...

[0]	validation_0-rmse:1.45007
[200]	validation_0-rmse:0.58734
[400]	validation_0-rmse:0.45165
[600]	validation_0-rmse:0.41707
[800]	validation_0-rmse:0.40500
[1000]	validation_0-rmse:0.40231
[1198]	validation_0-rmse:0.40142

YEKUN TRAIN RMSLE: 0.3535
YEKUN TEST RMSLE:  0.4014

Ən vacib 10 faktor:
CAT__Functioning Day_No         0.339748
CAT__Functioning Day_Yes        0.304148
CAT__Seasons_Winter             0.124390
NUM__Rainfall(mm)               0.040212
NUM__Temperature(°C)            0.031628
NUM__Humidity(%)                0.028693
CAT__Seasons_Summer             0.017839
NUM__hour_sin                   0.016939
CAT__Seasons_Autumn             0.015422
NUM__Solar Radiation (MJ/m2)    0.015133
dtype: float32
